### Chunking

In [ ]:
from typing import List

def split_into_chunks(doc_file: str) -> List[str]:
    with open(doc_file, 'r') as file:
        content = file.read()

    return [chunk for chunk in content.split("\n\n")]

chunks = split_into_chunks("doc.md")

for i, chunk in enumerate(chunks):
    print(f"[{i}] {chunk}\n")

### Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

# embedding_model = SentenceTransformer("shibing624/text2vec-base-chinese")
embedding_model = SentenceTransformer("BAAI/bge-large-en-v1.5")

def embed_chunk(chunk: str) -> List[float]:
    embedding = embedding_model.encode(chunk, normalize_embeddings=True)
    return embedding.tolist()

embeddings = [embed_chunk(chunk) for chunk in chunks]

print(len(embeddings))
print(embeddings[0])

### Vector DB

In [ ]:
import chromadb

chromadb_client = chromadb.EphemeralClient()

# 1. Delete the old collection if it exists to reset the dimensions
try:
    chromadb_client.delete_collection(name="default")
except:
    pass 

# 2. This will now create a fresh collection with 1024-dimension support
chromadb_collection = chromadb_client.get_or_create_collection(name="default")

# 3. Optimized batch-save function (much faster than a loop)
def save_embeddings(chunks: List[str], embeddings: List[List[float]]) -> None:
    chromadb_collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=[str(i) for i in range(len(chunks))]
    )

save_embeddings(chunks, embeddings)

#### use the following block to create a permanent vector database file, uncomment to run

In [ ]:
# import chromadb
# import os
# from typing import List

# # 1. Initialize a Persistent Client
# # This creates a folder named 'chroma_db' in your project directory to store the data
# db_path = os.path.join(os.getcwd(), "chroma_storage")
# chromadb_client = chromadb.PersistentClient(path=db_path)

# # 2. Get or create the collection
# # Since we aren't deleting it, this will load the existing data if it exists
# chromadb_collection = chromadb_client.get_or_create_collection(name="project_aether")

# def add_to_vector_db(chunks: List[str], embeddings: List[List[float]]) -> None:
#     """
#     Adds new content to the database. 
#     Uses timestamps or UUIDs for IDs to prevent overwriting existing entries.
#     """
#     import uuid
    
#     # Generate unique IDs so we don't overwrite old data
#     unique_ids = [str(uuid.uuid4()) for _ in range(len(chunks))]
    
#     chromadb_collection.add(
#         documents=chunks,
#         embeddings=embeddings,
#         ids=unique_ids
#     )
#     print(f"Successfully added {len(chunks)} new items to {db_path}")

# Example Usage:
# add_to_vector_db(chunks, embeddings)

### Retrieve

In [ ]:
def retrieve(query: str, top_k: int) -> List[str]:
    # Ensure there is actually data to query
    if chromadb_collection.count() == 0:
        print("Warning: The database is empty. Please add documents first.")
        return []
    query_embedding = embed_chunk(query)
    results = chromadb_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results['documents'][0]

query = "What is the required part number for the intake filters used in the Atacama Desert testing corridor, and how often must they be cleaned?"
retrieved_chunks = retrieve(query, 5)

for i, chunk in enumerate(retrieved_chunks):
    print(f"[{i}] {chunk}\n")

### Rerank

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L12-v2')

def rerank(query: str, retrieved_chunks: List[str], top_k: int) -> List[str]:
    # cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs)

    scored_chunks = list(zip(retrieved_chunks, scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    return [chunk for chunk, _ in scored_chunks][:top_k]

reranked_chunks = rerank(query, retrieved_chunks, 3)

for i, chunk in enumerate(reranked_chunks):
    print(f"[{i}] {chunk}\n")

### Testing

In [ ]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
google_client = genai.Client()

def generate(query: str, chunks: List[str]) -> str:
    # Pre-process chunks into a numbered list to avoid f-string join errors
    context_text = "\n\n".join([f"[Fragment {i+1}]: {chunk}" for i, chunk in enumerate(chunks)])
    
    prompt = f"""You are a highly precise technical assistant. Your task is to answer the user's question based ONLY on the provided document fragments.

    Guidelines:
    1. Use the provided context to form a factual and concise answer.
    2. If the answer is not contained within the context, state that you do not know. Do not hallucinate or use outside knowledge.
    3. Mention specific technical terms, part numbers, or metrics if they are relevant to the answer.

    Context Fragments:
    {context_text}

    User Question: {query}

    Answer:"""

    print(f"--- QUESTION: {query} ---\n")

    response = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

# --- Testing Questions ---
test_questions = [
    "What is the required part number for the intake filters used in the Atacama Desert testing corridor, and how often must they be cleaned?",
    "What specific logic does the drone default to if its three Node-Core units disagree on a flight path in software version 4.2?",
    "If the drone's external status LED shows a '3-short, 1-long' red flash pattern, what is the diagnosed issue and what is the required fix?"
]

# Loop through questions to test your RAG system
for i, q in enumerate(test_questions):
    print(f"\nTEST CASE {i+1}:")
    # Step 1: Retrieve (from your previous cells)
    retrieved = retrieve(q, 5) 
    # Step 2: Rerank (from your previous cells)
    reranked = rerank(q, retrieved, 3)
    # Step 3: Generate
    result = generate(q, reranked)
    print(f"RESULT:\n{result}\n" + "="*50)